In [77]:
import pandas as pd
import numpy as np

In [78]:
T = pd.read_csv("/home/joe/work/Fire/ML/New/mesh_forecast_T.csv")
Td = pd.read_csv("/home/joe/work/Fire/ML/New/mesh_forecast_Td.csv")
RH = pd.read_csv("/home/joe/work/Fire/ML/New/mesh_forecast_RH.csv")
pcp = pd.read_csv("/home/joe/work/Fire/ML/New/mesh_forecast_tp.csv")
VPD = pd.read_csv("/home/joe/work/Fire/ML/New/mesh_forecast_VPD.csv")
VPD3 = pd.read_csv("/home/joe/work/Fire/ML/New/mesh_forecast_VPD3.csv")
RH20 = pd.read_csv("/home/joe/work/Fire/ML/New/mesh_forecast_RH20.csv")


In [79]:
T = T[["point_id","lat","lon","year_month","lead","prediction"]]
T.rename(columns={"prediction": "T"}, inplace=True)
Td = Td[["point_id","lat","lon","year_month","lead","prediction"]]
Td.rename(columns={"prediction": "Td"}, inplace=True)
RH = RH[["point_id","lat","lon","year_month","lead","prediction"]]
RH.rename(columns={"prediction": "RH"}, inplace=True)
pcp = pcp[["point_id","lat","lon","year_month","lead","prediction"]]
pcp.rename(columns={"prediction": "pcp"}, inplace=True)
VPD = VPD[["point_id","lat","lon","year_month","lead","prediction"]]
VPD.rename(columns={"prediction": "VPD"}, inplace=True)
VPD3 = VPD3[["point_id","lat","lon","year_month","lead","prediction"]]
VPD3.rename(columns={"prediction": "VPD3"}, inplace=True)
RH20 = RH20[["point_id","lat","lon","year_month","lead","prediction"]]
RH20.rename(columns={"prediction": "RH20"}, inplace=True)

In [81]:
def getCenterPoints(R, latT,latB,lonL,lonR,STEP=5):
    # Grid definition
    lat_vals = np.round(np.arange(latT, latB -R, -R), 1)
    lon_vals = np.round(np.arange(lonL, lonR + R, R), 1)

    lat_core = lat_vals[2:-2]   # drop top 2 + bottom 2
    lon_core = lon_vals[2:-2]   # drop left 2 + right 2

    lat_centers = lat_core[::STEP]
    lon_centers = lon_core[::STEP]

    mesh_centers = []

    for lat in lat_centers:
        for lon in lon_centers:
            mesh_centers.append((lat, lon))

    return mesh_centers

def getMeshPoints(R, latT,latB,lonL,lonR,STEP=5):
    # Grid definition
    lat_vals = np.round(np.arange(latT, latB -R, -R), 1)
    lon_vals = np.round(np.arange(lonL, lonR + R, R), 1)

    lat_core = lat_vals[2:-2]   # drop top 2 + bottom 2
    lon_core = lon_vals[2:-2]   # drop left 2 + right 2

    lat_centers = lat_core[::STEP]
    lon_centers = lon_core[::STEP]

    mesh_centers = []

    for lat in lat_centers:
        for lon in lon_centers:
            mesh_centers.append((lat, lon))

    return mesh_centers

GRID_RADIUS_DEG = 0.21
LATT = 41.0
LATB = 37.0
LONL = -109.0+360
LONR = -102.0+360
R = .1
STEP = 5

centerPoints = getCenterPoints(R,LATT,LATB,LONL,LONR,STEP)

centerPoints 


[(np.float64(40.8), np.float64(251.2)),
 (np.float64(40.8), np.float64(251.7)),
 (np.float64(40.8), np.float64(252.2)),
 (np.float64(40.8), np.float64(252.7)),
 (np.float64(40.8), np.float64(253.2)),
 (np.float64(40.8), np.float64(253.7)),
 (np.float64(40.8), np.float64(254.2)),
 (np.float64(40.8), np.float64(254.7)),
 (np.float64(40.8), np.float64(255.2)),
 (np.float64(40.8), np.float64(255.7)),
 (np.float64(40.8), np.float64(256.2)),
 (np.float64(40.8), np.float64(256.7)),
 (np.float64(40.8), np.float64(257.2)),
 (np.float64(40.8), np.float64(257.7)),
 (np.float64(40.3), np.float64(251.2)),
 (np.float64(40.3), np.float64(251.7)),
 (np.float64(40.3), np.float64(252.2)),
 (np.float64(40.3), np.float64(252.7)),
 (np.float64(40.3), np.float64(253.2)),
 (np.float64(40.3), np.float64(253.7)),
 (np.float64(40.3), np.float64(254.2)),
 (np.float64(40.3), np.float64(254.7)),
 (np.float64(40.3), np.float64(255.2)),
 (np.float64(40.3), np.float64(255.7)),
 (np.float64(40.3), np.float64(256.2)),


In [82]:
from functools import reduce
dfs = [T, Td, RH, pcp, VPD, VPD3, RH20]
merged = reduce(lambda left, right: pd.merge(left, right, on=["point_id","lat","lon","year_month","lead"]), dfs)
merged['month'] = merged['year_month'].astype(str).str[-2:].astype(int)


In [83]:

latlons = pd.read_csv("/home/joe/work/Fire/ML/New/latlons_42x71.csv")

latlons['center'] = None

latlons['lon'] = round(360-abs(latlons['lon']),1)

for indx, (lat, lon) in enumerate(centerPoints):  
    latlons.loc[(np.abs(latlons.lat - lat) <= .21) &
        (np.abs(latlons.lon - lon) <= .21),'center']=indx
    
latlons.to_csv("/home/joe/work/Fire/ML/New/latlons_42x71_center.csv", index=False)

merged['center'] = None
for idx,row in latlons.iterrows():
    merged.loc[merged['point_id'] == row['point'],'center'] = row['center']


## Get Monthly Means

In [84]:
def load_era_global():
    con = duckdb.connect(ERA_DB)
    df = con.execute(f"SELECT * FROM {ERA_TABLE}").df()
    con.close()

    df["year_month"] = df["year_month"].astype(int)
    df["year"] = df["year_month"] // 100
    df["month"] = df["year_month"] % 100
    df["ym_index"] = df["year"] * 12 + df["month"]

    return df

In [86]:
import duckdb

path = "/home/joe/work/Fire/ML/Data"
path_clim = "/home/joe/work/Fire/ML/Data/Climate-Indices/New"

ERA_DB = f"{path}/DB/era5MonthlyMeansFinal.sqlite"
ERA_TABLE = "monthly_means"

df = load_era_global()

In [87]:
df.columns

Index(['point_id', 'year_month', 't_mean', 'td_mean', 'rh_mean', 'vpd_mean',
       'tp_sum', 'days_with_rh_lt_20', 'rh_lt_20_pct', 'vpd_gt_3_pct',
       'days_with_vpd_gt_3', 'year', 'month', 'ym_index'],
      dtype='object')

In [88]:
for idx,row in latlons.iterrows():
    df.loc[df['point_id'] == row['point'],'center'] = row['center']

In [89]:
df = df[df['year'] <= 2015]

In [90]:
df.shape

(930384, 15)

In [91]:
por_stats = df.groupby(['center','month'])[['t_mean', 'td_mean', 'rh_mean', 'tp_sum', 'vpd_mean',  'vpd_gt_3_pct', 'rh_lt_20_pct']].agg(['mean','std'])

In [92]:
por_stats['center'] = por_stats.index.get_level_values(0)
por_stats['month'] = por_stats.index.get_level_values(1)

In [93]:
por_stats.columns 

MultiIndex([(      't_mean', 'mean'),
            (      't_mean',  'std'),
            (     'td_mean', 'mean'),
            (     'td_mean',  'std'),
            (     'rh_mean', 'mean'),
            (     'rh_mean',  'std'),
            (      'tp_sum', 'mean'),
            (      'tp_sum',  'std'),
            (    'vpd_mean', 'mean'),
            (    'vpd_mean',  'std'),
            ('vpd_gt_3_pct', 'mean'),
            ('vpd_gt_3_pct',  'std'),
            ('rh_lt_20_pct', 'mean'),
            ('rh_lt_20_pct',  'std'),
            (      'center',     ''),
            (       'month',     '')],
           )

In [94]:
mapr = {'t_mean':'T',
 'td_mean':'Td',
 'rh_mean':'RH',
 'tp_sum':'pcp',
 'vpd_mean':'VPD',
 'vpd_gt_3_pct':'VPD3', 
    'rh_lt_20_pct':'RH20'}

In [95]:
# Convert por_stats MultiIndex columns -> flat monthly means by center/month
por_means = por_stats.copy()
por_means = por_means.rename(columns=mapr)

# Keep only the climatological mean level from the second column level
if isinstance(por_means.columns, pd.MultiIndex):
    por_means = por_means.xs('mean', axis=1, level=1)

# Bring center/month out of index (if still indexed)
if isinstance(por_means.index, pd.MultiIndex):
    por_means = por_means.reset_index()

# Rename DB variable names to anomaly variable names used in merged

# Build mean-column names to avoid collisions with anomaly columns in merged
vars_in_means = [v for v in por_means.columns]
por_means = por_means.rename(columns={v: f"{v}_mean" for v in vars_in_means})
por_means.rename(columns={"month_mean": "month", "center_mean": "center"}, inplace=True)
# Merge means onto anomaly table
join_cols = ['center', 'month']
mean_cols = [f"{v}_mean" for v in vars_in_means]
merged = merged.merge(
    por_means,
    on=join_cols,
    how='left'
)

# Add means back to anomalies -> absolute forecasts
for v in vars_in_means:
    if v in merged.columns and f"{v}_mean" in merged.columns:
        merged[f"{v}_abs"] = merged[v] + merged[f"{v}_mean"]

merged[[c for c in merged.columns if c.endswith('_abs')]].head()

,T_abs,Td_abs,RH_abs,pcp_abs,VPD_abs,VPD3_abs,RH20_abs
0,273.721384,267.284253,70.057218,0.031717,0.165032,-0.030795,0.010272
1,273.700138,267.319213,70.340796,0.037299,0.042244,-0.021573,0.004968
2,273.653254,267.291170,70.628506,0.037029,0.008460,-0.014856,-0.017351
3,273.608932,267.245226,70.920258,0.035191,-0.005447,-0.024739,-0.037344
4,273.503512,267.197787,71.204943,0.036936,0.003907,-0.020922,-0.064643


In [96]:
mean_cols

['center_mean',
 'month_mean',
 'T_mean',
 'Td_mean',
 'RH_mean',
 'pcp_mean',
 'VPD_mean',
 'VPD3_mean',
 'RH20_mean']

In [97]:
display (merged.head())

,point_id,lat,lon,year_month,lead,T,Td,RH,pcp,VPD,...,VPD_mean,VPD3_mean,RH20_mean,T_abs,Td_abs,RH_abs,pcp_abs,VPD_abs,VPD3_abs,RH20_abs
0,71,41.0,251.0,202603,1,2.263570,1.147378,0.306361,-0.002102,-0.042225,...,0.207257,0.0,0.002605,273.721384,267.284253,70.057218,0.031717,0.165032,-0.030795,0.010272
1,71,41.0,251.0,202603,2,2.242323,1.182338,0.589939,0.003479,-0.165013,...,0.207257,0.0,0.002605,273.700138,267.319213,70.340796,0.037299,0.042244,-0.021573,0.004968
2,71,41.0,251.0,202603,3,2.195439,1.154295,0.877649,0.003209,-0.198797,...,0.207257,0.0,0.002605,273.653254,267.291170,70.628506,0.037029,0.008460,-0.014856,-0.017351
3,71,41.0,251.0,202603,4,2.151118,1.108351,1.169401,0.001372,-0.212704,...,0.207257,0.0,0.002605,273.608932,267.245226,70.920258,0.035191,-0.005447,-0.024739,-0.037344
4,71,41.0,251.0,202603,5,2.045697,1.060912,1.454086,0.003116,-0.203350,...,0.207257,0.0,0.002605,273.503512,267.197787,71.204943,0.036936,0.003907,-0.020922,-0.064643


In [184]:

merged[['point_id','year_month','lead','T_abs', 'Td_abs', 'RH_abs', 'pcp_abs', 'VPD_abs', 'VPD3_abs',
       'RH20_abs']]

,point_id,year_month,lead,T_abs,Td_abs,RH_abs,pcp_abs,VPD_abs,VPD3_abs,RH20_abs
0,71,202603,1,273.721384,267.284253,70.057218,0.031717,0.165032,-0.030795,0.010272
1,71,202603,2,273.700138,267.319213,70.340796,0.037299,0.042244,-0.021573,0.004968
2,71,202603,3,273.653254,267.291170,70.628506,0.037029,0.008460,-0.014856,-0.017351
3,71,202603,4,273.608932,267.245226,70.920258,0.035191,-0.005447,-0.024739,-0.037344
4,71,202603,5,273.503512,267.197787,71.204943,0.036936,0.003907,-0.020922,-0.064643
...,...,...,...,...,...,...,...,...,...,...
33595,2909,202603,8,280.604174,269.609721,54.403762,0.032460,0.471049,0.008259,0.048183
33596,2909,202603,9,280.426958,269.620667,54.562358,0.035134,0.397736,0.008075,0.051199
33597,2909,202603,10,280.277076,269.616898,54.732496,0.036128,0.325212,-0.000526,0.049065
33598,2909,202603,11,280.136273,269.611954,54.903046,0.034540,0.269180,-0.008090,0.024927


In [58]:
# Compute forecast valid date as issue year_month + lead months
ym_str = (
    merged['year_month']
    .astype(str)
    .str.strip()
    .str.replace(r'\.0$', '', regex=True)
 )

# Parse common year_month encodings explicitly first
ym_dt = pd.to_datetime(ym_str, format='%Y%m', errors='coerce')
mask = ym_dt.isna()
if mask.any():
    ym_dt.loc[mask] = pd.to_datetime(ym_str[mask], format='%Y-%m', errors='coerce')

# Final fallback for other parseable formats (e.g., YYYY-MM-01)
mask = ym_dt.isna()
if mask.any():
    ym_dt.loc[mask] = pd.to_datetime(ym_str[mask], errors='coerce')

lead_m = pd.to_numeric(merged['lead'], errors='coerce').fillna(0).astype(int)
merged['valid_date'] = ((ym_dt.dt.to_period('M') + lead_m).dt.to_timestamp()).dt.date
merged['valid_year_month'] = pd.to_datetime(merged['valid_date']).dt.strftime('%Y-%m')

merged[['year_month', 'lead', 'valid_date', 'valid_year_month']].head(10)

,year_month,lead,valid_date,valid_year_month
0,202603,1,2026-04-01,2026-04
1,202603,2,2026-05-01,2026-05
2,202603,3,2026-06-01,2026-06
3,202603,4,2026-07-01,2026-07
4,202603,5,2026-08-01,2026-08
5,202603,6,2026-09-01,2026-09
6,202603,7,2026-10-01,2026-10
7,202603,8,2026-11-01,2026-11
8,202603,9,2026-12-01,2026-12
9,202603,10,2027-01-01,2027-01


In [155]:
df.rename(columns=mapr, inplace=True)   

In [64]:
merged[['point_id','lat', 'lon', 'year_month', 'lead', 'valid_date','T_abs',
       'Td_abs', 'RH_abs', 'pcp_abs', 'VPD_abs', 'VPD3_abs', 'RH20_abs',
       ]].to_csv("/home/joe/work/Fire/ML/New/merged_anomalies_with_means.csv", index=False)

In [63]:
merged['valid_date'].value_counts()

valid_date
2026-04-01    2800
2026-05-01    2800
2026-06-01    2800
2026-07-01    2800
2026-08-01    2800
2026-09-01    2800
2026-10-01    2800
2026-11-01    2800
2026-12-01    2800
2027-01-01    2800
2027-02-01    2800
2027-03-01    2800
Name: count, dtype: int64

In [2]:
df = pd.read_csv("merged_anomalies_with_means.csv")

In [3]:
df.shape

(33600, 13)

In [5]:
df.columns

Index(['point_id', 'lat', 'lon', 'year_month', 'lead', 'valid_date', 'T_abs',
       'Td_abs', 'RH_abs', 'pcp_abs', 'VPD_abs', 'VPD3_abs', 'RH20_abs'],
      dtype='object')

In [6]:
df.rename(columns={'T_abs':'T','Td_abs':'Td','RH_abs':'RH', 'pcp_abs':'pcp', 'VPD_abs':'VPD', 'VPD3_abs':'VPD3', 'RH20_abs':'RH20'}, inplace=True)

In [7]:
df.head()

,point_id,lat,lon,year_month,lead,valid_date,T,Td,RH,pcp,VPD,VPD3,RH20
0,71,41.0,251.0,202603,1,2026-04-01,273.721384,267.284253,70.057218,0.031717,0.165032,-0.030795,0.010272
1,71,41.0,251.0,202603,2,2026-05-01,273.700138,267.319213,70.340796,0.037299,0.042244,-0.021573,0.004968
2,71,41.0,251.0,202603,3,2026-06-01,273.653254,267.291170,70.628506,0.037029,0.008460,-0.014856,-0.017351
3,71,41.0,251.0,202603,4,2026-07-01,273.608932,267.245226,70.920258,0.035191,-0.005447,-0.024739,-0.037344
4,71,41.0,251.0,202603,5,2026-08-01,273.503512,267.197787,71.204943,0.036936,0.003907,-0.020922,-0.064643


In [ ]:
df.to_csv("/home/joe/work/Fire/ML/New/mesh_forecasts_with_means.csv", index=False)

## Winds

In [63]:
import duckdb
path="/home/joe/work/Fire/ML/New/Data/DB/"
ERA_DB = f"{path}era5_wind_means_29x17.sqlite"
#ERA_TABLE = "MEANS_TTdRHVPD"
ERA_TABLE = "wind_mean_100m_points"

con = duckdb.connect(ERA_DB)

dfO = con.execute(f"""
SELECT * FROM {ERA_TABLE}
""").df()

con.close()

In [64]:
df = dfO.copy()

In [66]:
dfO.columns

Index(['point_id', 'lat', 'lon', 'year_month', 'ws100', 'cnt_20mph',
       'cnt_25mph'],
      dtype='object')

In [67]:
dfO.rename(columns={'latitude':'lat','longitude':'lon','yrmo':'year_month'}, inplace=True)
dfO[['point_id', 'lat', 'lon', 'year_month','ws100', 'cnt_20mph',
       'cnt_25mph']].to_csv("/home/joe/work/Fire/ML/New/mesh_historical_winds_.csv", index=False  )

In [68]:
# Add 1 year to the datetime column
df_apr_2025_plus = df.loc[df["yrmo"] >= pd.Timestamp("2025-04-01")].copy()

df_apr_2025_plus["yrmo_plus_1y"] = df_apr_2025_plus["yrmo"] + pd.DateOffset(years=1)
df_apr_2025_plus[["yrmo", "yrmo_plus_1y"]].head()

,yrmo,yrmo_plus_1y
423,2025-04-01,2026-04-01
424,2025-05-01,2026-05-01
425,2025-06-01,2026-06-01
426,2025-07-01,2026-07-01
427,2025-08-01,2026-08-01


In [69]:
df_apr_2025_plus['yrmo'] = '2026-03'
df_apr_2025_plus.rename(columns={'yrmo_plus_1y': 'valid_date'}, inplace=True)


In [70]:
df_apr_2025_plus['lead'] = ((pd.to_datetime(df_apr_2025_plus['valid_date']) - pd.to_datetime(df_apr_2025_plus['yrmo'])).dt.days // 30).astype(int)          


In [71]:
df_apr_2025_plus['valid_date'] = df_apr_2025_plus['valid_date'].dt.date

In [72]:
df_apr_2025_plus['longitude'] = round(360-abs(df_apr_2025_plus['longitude']),2)
df_apr_2025_plus.head()


,point_id,latitude,longitude,yrmo,ws100,cnt_20mph,cnt_25mph,valid_date,lead
423,464,37.0,251.0,2026-03,9.810000,52,26,2026-04-01,1
424,464,37.0,251.0,2026-03,10.425806,52,12,2026-05-01,2
425,464,37.0,251.0,2026-03,8.930000,23,1,2026-06-01,3
426,464,37.0,251.0,2026-03,9.654839,31,8,2026-07-01,4
427,464,37.0,251.0,2026-03,8.461290,9,0,2026-08-01,5


In [73]:
df_apr_2025_plus.columns

Index(['point_id', 'latitude', 'longitude', 'yrmo', 'ws100', 'cnt_20mph',
       'cnt_25mph', 'valid_date', 'lead'],
      dtype='object')

In [74]:
df_apr_2025_plus.rename(columns={'latitude':'lat', 'longitude':'lon', 'yrmo':'year_month'}, inplace=True)

In [75]:
df_apr_2025_plus.head()

,point_id,lat,lon,year_month,ws100,cnt_20mph,cnt_25mph,valid_date,lead
423,464,37.0,251.0,2026-03,9.810000,52,26,2026-04-01,1
424,464,37.0,251.0,2026-03,10.425806,52,12,2026-05-01,2
425,464,37.0,251.0,2026-03,8.930000,23,1,2026-06-01,3
426,464,37.0,251.0,2026-03,9.654839,31,8,2026-07-01,4
427,464,37.0,251.0,2026-03,8.461290,9,0,2026-08-01,5


In [76]:
df_apr_2025_plus[['point_id', 'lat', 'lon', 'year_month','valid_date', 'lead','ws100', 'cnt_20mph',
       'cnt_25mph']].to_csv("/home/joe/work/Fire/ML/New/mesh_forecast_winds_.csv", index=False  )